# FaceInsight — Entraînement multi-tâches sur Colab

Prérequis (une seule fois) :
1. **Runtime GPU** : Exécution → Modifier le type d'exécution → T4 GPU (ou mieux avec Colab Pro+).
2. **Secrets Colab** (icône clé 🔑 dans la barre latérale) : ajouter `KAGGLE_USERNAME`, `KAGGLE_KEY` et `WANDB_API_KEY`, avec « Accès au notebook » activé.

Les checkpoints sont écrits sur Google Drive à **chaque epoch** : si la session Colab meurt, il suffit de relancer le notebook et d'exécuter la cellule « Reprise ».

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone https://github.com/mrSvet0zar/faceinsight.git 2>/dev/null || git -C faceinsight pull
%pip install -q wandb kaggle python-dotenv

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

# Datasets (~3 Go) — re-téléchargés à chaque session, le réseau Kaggle-Colab est rapide
%cd /content/faceinsight/backend
!python -m app.training.download_datasets
!python -m app.training.download_datasets ferplus
!python -m app.training.explore_datasets

## Entraînement

Pondérations de loss (`--w-emotion`, `--w-age`, `--w-gender`, `--w-facial-hair`, `--w-hair`)
à comparer entre runs dans W&B — donner un `--run-name` explicite à chaque config testée.

In [ ]:
CKPT_DIR = '/content/drive/MyDrive/faceinsight/checkpoints'

!python -m app.training.train \
  --epochs 20 --batch-size 128 --num-workers 4 \
  --out-dir {CKPT_DIR} \
  --run-name baseline-w-defaults

## Reprise après coupure de session

Ré-exécuter d'abord les cellules 1 à 3 (montage Drive, clone, datasets), puis :

In [ ]:
!python -m app.training.train \
  --epochs 20 --batch-size 128 --num-workers 4 \
  --out-dir {CKPT_DIR} \
  --run-name baseline-w-defaults --resume

## Run 3 — FER+ (labels réannotés) + crop cohérent train/inférence

`--emotion-dataset ferplus` active deux améliorations : les labels FER+
(vote majoritaire de 10 annotateurs, images contempt/unknown/NF écartées)
et un padding de contexte qui aligne la géométrie des crops d'entraînement
sur celle de l'inférence webcam (MediaPipe + marge de 35 %).

In [ ]:
!python -m app.training.train \
  --epochs 30 --batch-size 128 --num-workers 4 \
  --w-emotion 1.5 --w-age 0.15 \
  --emotion-dataset ferplus \
  --out-dir /content/drive/MyDrive/faceinsight/checkpoints-v3 \
  --run-name v3-ferplus-30ep

# Pour un checkpoint v3, ajouter --emotion-dataset ferplus (le test FER+
# n'est pas comparable au test FER2013 : labels différents)
!python -m app.training.evaluate \
  --checkpoint /content/drive/MyDrive/faceinsight/checkpoints-v3/best.pth \
  --emotion-dataset ferplus \
  --json /content/drive/MyDrive/faceinsight/checkpoints-v3/eval_report.json

In [ ]:
!python -m app.training.evaluate \
  --checkpoint {CKPT_DIR}/best.pth \
  --json {CKPT_DIR}/eval_report.json